# 🐍 Python od podstaw — Moduł 10: API, `requests` i wprowadzenie do `pandas`

### Ostatni przystanek — dane z internetu i ich analiza

To ostatni moduł tej serii, i celowo łączy w sobie prawie wszystko: pobieranie danych z
sieci (`requests`), ich strukturę (JSON — pamiętasz moduł 7?), obsługę błędów sieciowych
(moduł 5), i na koniec pierwszy kontakt z `pandas` — biblioteką, bez której trudno
wyobrazić sobie analizę danych w Pythonie.

**Uwaga:** ten moduł (w przeciwieństwie do poprzednich) wymaga połączenia z internetem —
komórki pobierające dane z API zadziałają tylko wtedy, gdy Twój komputer ma dostęp do
sieci.

## Spis treści

1. [Co to jest API — szybkie przypomnienie](#sec1)
2. [Biblioteka `requests` — pierwsze zapytanie](#sec2)
3. [Obsługa błędów sieciowych](#sec3)
4. [`pandas` — pierwszy kontakt](#sec4)
5. [Wczytywanie CSV do `pandas`](#sec5)
6. [Filtrowanie i sortowanie](#sec6)
7. [`groupby` — agregacja danych](#sec7)
8. [Łączenie `requests` z `pandas`](#sec8)
9. [Ciekawostka: skąd nazwy „pandas” i „requests”](#sec9)
10. [Podsumowanie całej serii](#sec10)
11. [Ćwiczenia](#sec11)

---

<a id="sec1"></a>
## 1. Co to jest API — szybkie przypomnienie

**API** (*Application Programming Interface*) to sposób, w jaki jeden program może
poprosić inny o dane albo zlecić mu jakąś akcję — najczęściej przez internet, wysyłając
zapytanie na konkretny adres (URL) i dostając odpowiedź, zwykle w formacie **JSON**
(pamiętasz moduł 7 — `import json`?). W tym module użyjemy darmowego, publicznego API
testowego: `jsonplaceholder.typicode.com` — stworzonego właśnie po to, żeby się na nim
uczyć.

<a id="sec2"></a>
## 2. Biblioteka `requests` — pierwsze zapytanie

`requests` to najpopularniejszy pakiet do wykonywania zapytań HTTP w Pythonie (nie
jest częścią biblioteki standardowej - `pip install requests`, choć w Anacondzie jest
zwykle już zainstalowany). `requests.get(url)` wysyła zapytanie typu GET i zwraca
obiekt odpowiedzi.

In [ ]:
import requests

odpowiedz = requests.get("https://jsonplaceholder.typicode.com/users/1")

print(odpowiedz.status_code)   # 200 = sukces
print(odpowiedz.headers["Content-Type"])

dane = odpowiedz.json()        # zamienia treść odpowiedzi (JSON) na słownik Pythona
print(dane)
print(dane["name"], dane["email"])

> 💡 **Kody statusu HTTP**
>
> 200 to sukces, ale warto znać też inne częste kody: `404` (nie znaleziono zasobu), `401`/`403` (brak autoryzacji/dostępu), `500` (błąd po stronie serwera). `response.ok` to skrót zwracający `True` dla kodów 200-399.

<a id="sec3"></a>
## 3. Obsługa błędów sieciowych

Zapytanie sieciowe może się nie udać na wiele sposobów: brak internetu, zły adres URL,
serwer nie odpowiada w rozsądnym czasie. Warto to obsłużyć - dokładnie tymi samymi
narzędziami z modułu 5 (`try`/`except`).

In [ ]:
import requests

try:
    odpowiedz = requests.get(
        "https://jsonplaceholder.typicode.com/users/1",
        timeout=5   # poddaj się po 5 sekundach, zamiast czekać w nieskończoność
    )
    odpowiedz.raise_for_status()   # zgłasza wyjątek, jeśli status to błąd (4xx/5xx)
    print(odpowiedz.json())
except requests.exceptions.Timeout:
    print("Serwer nie odpowiedział na czas")
except requests.exceptions.ConnectionError:
    print("Brak połączenia z internetem albo zły adres")
except requests.exceptions.HTTPError as blad:
    print(f"Serwer zwrócił błąd: {blad}")

> ⚠️ **`raise_for_status()` to nie automat**
>
> `requests` NIE zgłasza wyjątku samo z siebie, gdy serwer zwróci błąd (np. 404) - `response.status_code` po prostu będzie równy 404, a `response.ok` da `False`, ale kod pójdzie dalej normalnie. Trzeba jawnie wywołać `raise_for_status()`, żeby zamienić błędny status na wyjątek Pythona, który można złapać `except`-em.

<a id="sec4"></a>
## 4. `pandas` — pierwszy kontakt

`pandas` to biblioteka do pracy z danymi tabelarycznymi - jej głównym obiektem jest
**DataFrame**, czyli coś na kształt arkusza kalkulacyjnego wewnątrz Pythona: wiersze,
kolumny, każda kolumna ma swój typ danych.

In [ ]:
import pandas as pd

produkty = [
    {"nazwa": "Chleb", "cena": 4.5, "ilosc": 20},
    {"nazwa": "Mleko", "cena": 3.2, "ilosc": 15},
    {"nazwa": "Jajka", "cena": 12.0, "ilosc": 8},
]

df = pd.DataFrame(produkty)
print(df)
print("---")
print(df.head(2))       # pierwsze 2 wiersze
print("---")
print(df.describe())    # podstawowe statystyki kolumn liczbowych
print("---")
print(df["cena"])       # dostęp do pojedynczej kolumny
print(df["cena"].mean())   # średnia cena

> 💡 **Ciekawostka**
>
> Import `pandas` niemal zawsze robi się pod aliasem `pd` (`import pandas as pd`) - to konwencja tak silna, że każdy kod pandas na świecie jej używa. Podobnie `numpy` importuje się jako `np`. Warto się do tego od razu przyzwyczaić.

<a id="sec5"></a>
## 5. Wczytywanie CSV do `pandas`

`pandas` ma wbudowaną funkcję do wczytywania CSV bezpośrednio do DataFrame - znacznie
wygodniejszą niż ręczna pętla po `csv.DictReader` z modułu 8.

In [ ]:
import pandas as pd
import csv

# Najpierw stwórzmy plik CSV (jak w module 8):
with open("produkty.csv", "w", newline="", encoding="utf-8") as plik:
    zapisujacy = csv.writer(plik)
    zapisujacy.writerow(["nazwa", "cena", "ilosc"])
    zapisujacy.writerow(["Chleb", 4.5, 20])
    zapisujacy.writerow(["Mleko", 3.2, 15])
    zapisujacy.writerow(["Jajka", 12.0, 8])
    zapisujacy.writerow(["Kawa", 25.0, 5])

df = pd.read_csv("produkty.csv")
print(df)
print(df.dtypes)   # typ danych każdej kolumny, wykryty automatycznie

<a id="sec6"></a>
## 6. Filtrowanie i sortowanie

Filtrowanie wierszy w `pandas` wygląda inaczej niż `if` w pętli - używa się
**maski logicznej**: warunku, który dla każdego wiersza daje `True`/`False`, a potem
tej maski jako "indeksu".

In [ ]:
import pandas as pd

df = pd.read_csv("produkty.csv")

drogie = df[df["cena"] > 5]           # maska logiczna - tylko wiersze z ceną > 5
print(drogie)

posortowane = df.sort_values("cena", ascending=False)   # sortowanie malejąco
print(posortowane)

print(df[df["cena"] > 5]["nazwa"])    # filtr + wybór jednej kolumny

<a id="sec7"></a>
## 7. `groupby` — agregacja danych

`groupby` grupuje wiersze według wartości w kolumnie i pozwala policzyć zagregowaną
statystykę dla każdej grupy naraz - to odpowiednik ręcznego słownika z licznikami z
modułu 3, tylko w jednej linii.

In [ ]:
import pandas as pd

zamowienia = pd.DataFrame([
    {"klient": "Kamil", "suma": 50},
    {"klient": "Ania", "suma": 30},
    {"klient": "Kamil", "suma": 20},
    {"klient": "Tomek", "suma": 100},
    {"klient": "Ania", "suma": 15},
])

suma_na_klienta = zamowienia.groupby("klient")["suma"].sum()
print(suma_na_klienta)

print("---")
print(zamowienia.groupby("klient")["suma"].agg(["sum", "mean", "count"]))

<a id="sec8"></a>
## 8. Łączenie `requests` z `pandas`

Typowy przepływ pracy: pobierz dane z API (lista słowników w JSON), zbuduj z nich
DataFrame, analizuj dalej narzędziami `pandas`.

In [ ]:
import requests
import pandas as pd

odpowiedz = requests.get("https://jsonplaceholder.typicode.com/users", timeout=5)
odpowiedz.raise_for_status()
uzytkownicy = odpowiedz.json()   # lista słowników

df = pd.DataFrame(uzytkownicy)
print(df.columns.tolist())         # jakie kolumny przyszły z API
print(df[["name", "email", "phone"]].head())

<a id="sec9"></a>
## 9. Ciekawostka: skąd nazwy „pandas” i „requests”

Dwie zupełnie różne historie nazewnicze.

> 💡 **Ciekawostka**
>
> Nazwa `pandas` nie pochodzi od zwierzęcia - to skrót od «panel data» (dane panelowe), terminu z ekonometrii oznaczającego wielowymiarowe zestawy danych. `requests` z kolei ma nieoficjalne motto «HTTP for Humans» (HTTP dla ludzi) - został stworzony, bo wbudowana w Pythona alternatywa (`urllib`) była uznawana za nieczytelną i niewygodną w użyciu.

<a id="sec10"></a>
## 10. Podsumowanie całej serii

Po tym module powinno być jasne:

- jak wysyłać zapytania GET przez `requests` i odczytywać odpowiedź JSON,
- jak obsługiwać błędy sieciowe (`timeout`, `ConnectionError`, `raise_for_status()`),
- jak stworzyć DataFrame z listy słowników albo pliku CSV,
- podstawowe operacje: `.head()`, `.describe()`, filtrowanie maską logiczną,
  `.sort_values()`,
- jak agregować dane przez `.groupby()`.

To zamyka dziesięciomodułową serię: od `print("Witaj, świecie!")` po pobieranie danych
z internetu i ich analizę. Po drodze było wszystko, co potrzebne do pisania realnych
programów: zmienne, sterowanie przepływem, kolekcje, funkcje, obsługa błędów i plików,
programowanie obiektowe, organizacja kodu w moduły, przetwarzanie tekstu, testowanie, i
na końcu - praca z danymi z zewnętrznego świata.

<a id="sec11"></a>
## 11. Ćwiczenia

Ostatni komplet ćwiczeń w całej serii - ostatnie zadanie to mały projekt spinający
kilka modułów naraz.

> 📝 **Ćwiczenie 1: Pojedynczy użytkownik z API**
>
> Pobierz dane użytkownika o id `3` z `https://jsonplaceholder.typicode.com/users/3` i wypisz jego imię, nazwę firmy (`company.name` w zwróconym słowniku) oraz miasto (`address.city`).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import requests

odpowiedz = requests.get("https://jsonplaceholder.typicode.com/users/3", timeout=5)
uzytkownik = odpowiedz.json()

print(uzytkownik["name"])
print(uzytkownik["company"]["name"])
print(uzytkownik["address"]["city"])
```
</details>

> 📝 **Ćwiczenie 2: Liczenie postów danego użytkownika**
>
> Pobierz wszystkie posty z `https://jsonplaceholder.typicode.com/posts` (to lista słowników, każdy z kluczem `userId`). Policz, ile postów należy do użytkownika o `userId == 1`, bez użycia pandas - zwykłą pętlą `for`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import requests

odpowiedz = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=5)
posty = odpowiedz.json()

licznik = 0
for post in posty:
    if post["userId"] == 1:
        licznik += 1

print(f"Użytkownik 1 ma {licznik} postów")
```
</details>

> 📝 **Ćwiczenie 3: Bezpieczne zapytanie z pełną obsługą błędów**
>
> Napisz funkcję `pobierz_dane(url)`, która wykonuje zapytanie GET z `timeout=5`, wywołuje `raise_for_status()`, i w bloku `try`/`except` obsługuje `Timeout`, `ConnectionError` oraz `HTTPError`, zwracając `None` i wypisując komunikat w każdym z tych przypadków, a w przeciwnym razie zwraca `response.json()`. Przetestuj na poprawnym adresie oraz na celowo błędnym (np. `https://jsonplaceholder.typicode.com/users/99999`).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import requests

def pobierz_dane(url):
    try:
        odpowiedz = requests.get(url, timeout=5)
        odpowiedz.raise_for_status()
        return odpowiedz.json()
    except requests.exceptions.Timeout:
        print("Serwer nie odpowiedział na czas")
        return None
    except requests.exceptions.ConnectionError:
        print("Brak połączenia z internetem")
        return None
    except requests.exceptions.HTTPError as blad:
        print(f"Błąd HTTP: {blad}")
        return None

print(pobierz_dane("https://jsonplaceholder.typicode.com/users/1"))
print(pobierz_dane("https://jsonplaceholder.typicode.com/users/99999"))
```

Podpowiedź: `/users/99999` zwróci pusty obiekt `{}` (nie błąd 404) w tym konkretnym API testowym - to dobra okazja, żeby zauważyć, że różne API różnie sygnalizują «nie znaleziono», więc zawsze warto sprawdzić dokumentację konkretnego API, z którego korzystasz.
</details>

> 📝 **Ćwiczenie 4: DataFrame z inwentarza (moduł 3/6)**
>
> Wróć do listy produktów z modułu 3/6 (`nazwa`, `cena`, `ilosc`). Zbuduj z niej DataFrame, dodaj nową kolumnę `wartosc = cena * ilosc` (przypisanie do nowej kolumny działa jak do klucza słownika: `df['wartosc'] = ...`), i wypisz `df.describe()`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import pandas as pd

produkty = [
    {"nazwa": "Chleb", "cena": 4.5, "ilosc": 20},
    {"nazwa": "Mleko", "cena": 3.2, "ilosc": 15},
    {"nazwa": "Jajka", "cena": 12.0, "ilosc": 8},
]

df = pd.DataFrame(produkty)
df["wartosc"] = df["cena"] * df["ilosc"]
print(df)
print(df.describe())
```
</details>

> 📝 **Ćwiczenie 5: Filtrowanie i sortowanie DataFrame**
>
> Mając DataFrame z poprzedniego ćwiczenia, wypisz tylko produkty z `wartosc > 50`, posortowane malejąco po `wartosc`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import pandas as pd

produkty = [
    {"nazwa": "Chleb", "cena": 4.5, "ilosc": 20},
    {"nazwa": "Mleko", "cena": 3.2, "ilosc": 15},
    {"nazwa": "Jajka", "cena": 12.0, "ilosc": 8},
]

df = pd.DataFrame(produkty)
df["wartosc"] = df["cena"] * df["ilosc"]

wynik = df[df["wartosc"] > 50].sort_values("wartosc", ascending=False)
print(wynik)
```
</details>

> 📝 **Ćwiczenie 6: `groupby` na zamówieniach**
>
> Mając listę zamówień (słowniki z kluczami `klient` i `suma`, kilku klientów powtarzających się), zbuduj DataFrame i policz przez `.groupby()` łączną sumę zamówień na klienta, posortowaną malejąco.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import pandas as pd

zamowienia = pd.DataFrame([
    {"klient": "Kamil", "suma": 50},
    {"klient": "Ania", "suma": 30},
    {"klient": "Kamil", "suma": 20},
    {"klient": "Tomek", "suma": 100},
    {"klient": "Ania", "suma": 15},
])

wynik = zamowienia.groupby("klient")["suma"].sum().sort_values(ascending=False)
print(wynik)
```
</details>

> 📝 **Ćwiczenie 7: API + pandas razem**
>
> Pobierz listę użytkowników z `https://jsonplaceholder.typicode.com/users`, zbuduj z niej DataFrame zawierający tylko kolumny `name`, `email` i `city` (podpowiedź: `city` trzeba wyciągnąć ręcznie z zagnieżdżonego `address` PRZED stworzeniem DataFrame, np. list comprehension budującą nową listę słowników z płaskimi kluczami).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import requests
import pandas as pd

odpowiedz = requests.get("https://jsonplaceholder.typicode.com/users", timeout=5)
uzytkownicy = odpowiedz.json()

splaszczeni = [
    {"name": u["name"], "email": u["email"], "city": u["address"]["city"]}
    for u in uzytkownicy
]

df = pd.DataFrame(splaszczeni)
print(df)
```
</details>

> 🔥 **Ćwiczenie 8 (wyzwanie): Mini-raport z API do CSV**
>
> Pobierz wszystkie posty z `https://jsonplaceholder.typicode.com/posts` w bloku `try`/`except` (obsłuż błędy sieciowe jak w ćwiczeniu 3). Zbuduj DataFrame, policz przez `.groupby('userId')` liczbę postów każdego użytkownika (podpowiedź: `.size()` zamiast `.sum()`), posortuj malejąco po liczbie postów, i zapisz wynik do pliku `raport_postow.csv` przez `.to_csv()`. Na koniec wczytaj ten plik z powrotem `pd.read_csv()`, żeby potwierdzić, że zapis się udał.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import requests
import pandas as pd

try:
    odpowiedz = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=5)
    odpowiedz.raise_for_status()
    posty = odpowiedz.json()
except requests.exceptions.RequestException as blad:
    print(f"Błąd pobierania danych: {blad}")
    posty = []

if posty:
    df = pd.DataFrame(posty)
    raport = df.groupby("userId").size().sort_values(ascending=False)
    raport.name = "liczba_postow"
    raport.to_csv("raport_postow.csv")

    wczytany = pd.read_csv("raport_postow.csv")
    print(wczytany)
```

Podpowiedź: `requests.exceptions.RequestException` to wspólna klasa bazowa dla `Timeout`, `ConnectionError` i `HTTPError` - łapanie jej jednym `except` to wygodny skrót, gdy nie potrzebujesz różnych komunikatów dla każdego typu błędu osobno, tylko chcesz bezpiecznie obsłużyć «cokolwiek poszło nie tak z siecią».
</details>

---

### To już koniec tej serii

Dziesięć modułów, od pierwszego `print()` po pobieranie i analizowanie danych z
internetu. Naturalne kolejne kroki stąd, w zależności od tego, gdzie chcesz iść dalej:
wykresy i wizualizacja danych (`matplotlib`, `seaborn`), głębsze `pandas` (łączenie
tabel, czyszczenie brudnych danych), budowanie prostych aplikacji webowych (`Flask`,
`FastAPI`), albo uczenie maszynowe (`scikit-learn`). Gratulacje za przejście całej
ścieżki — to solidna podstawa na start.